# Mạng Residual (Residual Networks)

Chào mừng bạn đến với bài tập thứ hai của tuần này! Bạn sẽ học cách xây dựng các mạng tích chập (CNN) rất sâu bằng cách sử dụng Mạng Residual (ResNet). Về mặt lý thuyết, các mạng rất sâu có khả năng biểu diễn những hàm số cực kỳ phức tạp; tuy nhiên, trên thực tế, chúng lại rất khó huấn luyện. Mạng Residual, được giới thiệu bởi [He và cộng sự](https://arxiv.org/pdf/1512.03385.pdf), cho phép huấn luyện các mạng có độ sâu lớn hơn nhiều so với những gì có thể thực hiện được trước đây.

**Trong bài tập này, bạn sẽ:**
- Triển khai các khối xây dựng cơ bản của ResNet.
- Kết hợp các khối này để xây dựng và huấn luyện một mạng thần kinh tiên tiến (state-of-the-art) phục vụ bài toán phân loại hình ảnh.

## <font color='darkblue'>Các cập nhật</font>

#### Nếu bạn đang làm việc trên notebook trước khi có bản cập nhật này...
* Notebook hiện tại là phiên bản "2a".
* Bạn có thể tìm thấy phần công việc ban đầu của mình đã được lưu trong notebook với tên phiên bản cũ ("v2").
* Để xem danh sách tệp, hãy vào menu "File->Open"; thao tác này sẽ mở một thẻ (tab) mới hiển thị danh sách các tệp.

#### Danh sách các cập nhật
* Đối với việc kiểm thử trên ảnh, đã thay thế `preprocess_input(x)` bằng `x=x/255.0` để chuẩn hóa ảnh đầu vào theo cùng cách mà dữ liệu huấn luyện của mô hình đã được chuẩn hóa.
* Sử dụng thuật ngữ các lớp "nông hơn" (shallower) để chỉ những lớp gần với đầu vào, và các lớp "sâu hơn" (deeper) để chỉ những lớp gần với đầu ra (thay vì dùng từ "thấp hơn" hay "sớm hơn").
* Đã thêm/cập nhật các hướng dẫn.

This assignment will be done in Keras. 

Before jumping into the problem, let's run the cell below to load the required packages.

In [3]:
!pip install pydot
import numpy as np
import tensorflow as tf
import keras

from keras import layers
from keras.layers import (
    Input, Add, Dense, Activation, ZeroPadding2D,
    BatchNormalization, Flatten, Conv2D,
    AveragePooling2D, MaxPooling2D, GlobalMaxPooling2D
)

from keras.models import Model, load_model
from keras.preprocessing import image
from keras.utils import get_file, model_to_dot, plot_model
from keras.applications.resnet50 import preprocess_input
from keras.initializers import glorot_uniform

import pydot
from IPython.display import SVG
from PIL import Image
from matplotlib.pyplot import imshow

import keras.backend as K

# Cấu hình định dạng ảnh
keras.config.set_image_data_format('channels_last')

# Thư viện hỗ trợ bài ResNet
from resnets_utils import *

%matplotlib inline

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


I0000 00:00:1791767952.872220 2959807 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow: 2.21.0
Keras: 3.15.1
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 1 - Vấn đề của các mạng nơ-ron rất sâu

Tuần trước, bạn đã xây dựng mạng nơ-ron tích chập (CNN) đầu tiên của mình. Trong những năm gần đây, các mạng nơ-ron ngày càng trở nên sâu hơn; các mạng tiên tiến nhất đã phát triển từ chỗ chỉ có vài lớp (ví dụ: AlexNet) lên đến hơn một trăm lớp.

*   Lợi ích chính của một mạng rất sâu là khả năng biểu diễn các hàm số cực kỳ phức tạp. Nó cũng có thể học các đặc trưng ở nhiều cấp độ trừu tượng khác nhau, từ các đường biên (ở những lớp nông hơn, gần với đầu vào) cho đến những đặc trưng rất phức tạp (ở những lớp sâu hơn, gần với đầu ra).
*   Tuy nhiên, việc sử dụng mạng sâu hơn không phải lúc nào cũng mang lại hiệu quả. Một rào cản lớn trong quá trình huấn luyện chúng là hiện tượng triệt tiêu gradient (vanishing gradients): ở các mạng rất sâu, tín hiệu gradient thường giảm nhanh về mức 0, khiến cho thuật toán hạ gradient (gradient descent) trở nên cực kỳ chậm chạp.
*   Cụ thể hơn, trong quá trình hạ gradient, khi thực hiện lan truyền ngược (backpropagation) từ lớp cuối cùng về lớp đầu tiên, bạn phải nhân với ma trận trọng số tại mỗi bước; do đó, giá trị gradient có thể giảm theo cấp số nhân về 0 (hoặc trong những trường hợp hiếm gặp, nó có thể tăng theo cấp số nhân và "bùng nổ" - exploding gradient - để đạt các giá trị rất lớn).
*   Vì vậy, trong quá trình huấn luyện, bạn có thể thấy độ lớn (hay chuẩn - norm) của gradient tại các lớp nông giảm rất nhanh về 0 khi quá trình huấn luyện diễn ra:

<img src="images/vanishing_grad_kiank.png" style="width:450px;height:220px;">
<caption><center> <u> <font color='purple'> **Figure 1** </u><font color='purple'>  : **Vanishing gradient** <br> The speed of learning decreases very rapidly for the shallower layers as the network trains </center></caption>

You are now going to solve this problem by building a Residual Network!

## 2 - Building a Residual Network

In ResNets, a "shortcut" or a "skip connection" allows the model to skip layers:  

<img src="images/skip_connection_kiank.png" style="width:650px;height:200px;">
<caption><center> <u> <font color='purple'> **Figure 2** </u><font color='purple'>  : A ResNet block showing a **skip-connection** <br> </center></caption>



Hình bên trái hiển thị "đường dẫn chính" đi qua mạng. Hình bên phải bổ sung một đường tắt vào đường dẫn chính đó. Bằng cách xếp chồng các khối ResNet này lên nhau, bạn có thể tạo ra một mạng rất sâu.

Trong bài giảng, chúng ta cũng đã thấy rằng việc sử dụng các khối ResNet có đường tắt giúp một khối bất kỳ dễ dàng học được hàm đồng nhất (identity function). Điều này có nghĩa là bạn có thể xếp chồng thêm các khối ResNet mà ít có nguy cơ làm giảm hiệu suất trên tập huấn luyện.

(Ngoài ra còn có bằng chứng cho thấy khả năng dễ dàng học hàm đồng nhất đóng góp vào hiệu suất vượt trội của ResNet nhiều hơn cả vai trò của kết nối bỏ qua trong việc khắc phục vấn đề triệt tiêu gradient).

Có hai loại khối chính được sử dụng trong ResNet, tùy thuộc chủ yếu vào việc kích thước đầu vào và đầu ra có giống nhau hay khác nhau. Bạn sẽ triển khai cả hai loại này: "khối đồng nhất" (identity block) và "khối tích chập" (convolutional block).

### 2.1 - Khối identity (Identity block)

Khối identity là khối tiêu chuẩn được sử dụng trong các mạng ResNet, tương ứng với trường hợp mà giá trị kích hoạt đầu vào (ví dụ: $a^{[l]}$) có cùng kích thước với giá trị kích hoạt đầu ra (ví dụ: $a^{[l+2]}$). Để làm rõ các bước diễn ra bên trong khối identity của ResNet, dưới đây là một sơ đồ khác minh họa chi tiết từng bước:

<img src="images/idblock2_kiank.png" style="width:650px;height:150px;">
<caption><center> <u> <font color='purple'> **Hình 3** </u><font color='purple'>  : **Khối identity.** Kết nối tắt (skip connection) "bỏ qua" 2 lớp. </center></caption>

Đường dẫn phía trên là "đường tắt" (shortcut path). Đường dẫn phía dưới là "đường dẫn chính" (main path). Trong sơ đồ này, chúng tôi cũng thể hiện rõ các bước CONV2D và ReLU ở mỗi lớp. Để đẩy nhanh quá trình huấn luyện, chúng tôi cũng đã thêm bước BatchNorm. Đừng lo lắng về việc triển khai nó sẽ phức tạp—bạn sẽ thấy rằng BatchNorm chỉ tốn một dòng mã trong Keras!

Trong bài tập này, bạn sẽ thực sự triển khai một phiên bản mạnh mẽ hơn một chút của khối identity này, trong đó kết nối tắt sẽ "bỏ qua" 3 lớp ẩn thay vì 2 lớp. Nó có dạng như sau:

<img src="images/idblock3_kiank.png" style="width:650px;height:150px;">
<caption><center> <u> <font color='purple'> **Hình 4** </u><font color='purple'>  : **Khối identity.** Kết nối tắt "bỏ qua" 3 lớp.</center></caption>

Dưới đây là các bước cụ thể.

Thành phần đầu tiên của nhánh chính:
- Lớp CONV2D thứ nhất có $F_1$ bộ lọc (filter) với kích thước (1,1) và bước trượt (stride) là (1,1). Chế độ padding là "valid" và tên lớp là `conv_name_base + '2a'`. Sử dụng giá trị 0 làm seed cho quá trình khởi tạo ngẫu nhiên.
- Lớp BatchNorm thứ nhất thực hiện chuẩn hóa theo trục 'channels' (kênh). Tên lớp là `bn_name_base + '2a'`.
- Sau đó áp dụng hàm kích hoạt ReLU. Lớp này không có tên và không có tham số siêu cấp (hyperparameter).

Thành phần thứ hai của nhánh chính:
- Lớp CONV2D thứ hai có $F_2$ bộ lọc với kích thước $(f,f)$ và bước trượt là (1,1). Chế độ padding là "same" và tên lớp là `conv_name_base + '2b'`. Sử dụng giá trị 0 làm seed cho quá trình khởi tạo ngẫu nhiên.
- Lớp BatchNorm thứ hai thực hiện chuẩn hóa theo trục 'channels'. Tên lớp là `bn_name_base + '2b'`.
- Sau đó áp dụng hàm kích hoạt ReLU. Lớp này không có tên và không có tham số siêu cấp.

Thành phần thứ ba của nhánh chính:
- Lớp CONV2D thứ ba có $F_3$ bộ lọc với kích thước (1,1) và bước trượt là (1,1). Chế độ padding là "valid" và tên lớp là `conv_name_base + '2c'`. Sử dụng giá trị 0 làm seed cho quá trình khởi tạo ngẫu nhiên.
- Lớp BatchNorm thứ ba thực hiện chuẩn hóa theo trục 'channels'. Tên lớp là `bn_name_base + '2c'`.
- Lưu ý rằng **không** có hàm kích hoạt ReLU trong thành phần này.

Bước cuối cùng:
- Cộng `X_shortcut` và đầu ra từ lớp thứ ba `X` lại với nhau.
- **Gợi ý**: Cú pháp sẽ có dạng `Add()([var1,var2])`
- Sau đó áp dụng hàm kích hoạt ReLU. Lớp này không có tên và không có tham số siêu cấp. **Bài tập**: Triển khai khối identity (identity block) của ResNet. Chúng tôi đã triển khai thành phần đầu tiên của nhánh chính (main path). Hãy đọc kỹ phần này để đảm bảo bạn hiểu rõ cơ chế hoạt động của nó. Bạn cần triển khai các phần còn lại.
- Để triển khai bước Conv2D: [Conv2D](https://keras.io/layers/convolutional/#conv2d)
- Để triển khai BatchNorm: [BatchNormalization](https://faroit.github.io/keras-docs/1.2.2/layers/normalization/) (tham số `axis`: Số nguyên, xác định trục cần chuẩn hóa - thường là trục 'channels')
- Đối với hàm kích hoạt (activation), hãy sử dụng: `Activation('relu')(X)`
- Để cộng giá trị được truyền qua nhánh tắt (shortcut): [Add](https://keras.io/layers/merge/#add)

In [ ]:
#identity_block: Input và output có cùng kích thước, nên shortcut giữ nguyên input.
def identity_block(X , f , filters , stage , block) :
    """
    Triển khai khối identity (identity block) như được mô tả trong Hình 4

    Các tham số:
    X -- tensor đầu vào có kích thước (m, n_H_prev, n_W_prev, n_C_prev)
    f -- số nguyên, xác định kích thước cửa sổ của lớp CONV ở giữa trên nhánh chính
    filters -- danh sách các số nguyên (kiểu Python), xác định số lượng bộ lọc (filters) trong các lớp CONV của nhánh chính
    stage -- số nguyên, dùng để đặt tên cho các lớp dựa trên vị trí của chúng trong mạng
    block -- chuỗi/ký tự, dùng để đặt tên cho các lớp dựa trên vị trí của chúng trong mạng

    Giá trị trả về:
    X -- đầu ra của khối identity, tensor có kích thước (n_H, n_W, n_C)
    """
    # defining name basis
    conv_name_base = 'res' + str(stage)+ block + '_branch'#tên cơ sở của các lớp Convolution
    bn_name_base = 'bn'+str(stage) + block + '_branch' # tên cơ sở của các lớp BatchNormalization
    
    # Lấy các bộ lọc
    F1 , F2 , F3 = filters # Số filters quyết định số kênh đầu ra của lớp convolution.

    # Lưu giá trị đầu vào. Bạn sẽ cần nó sau này để thêm lại vào đường dẫn chính.
    X_shortcut = X
    
    # Thành phần đầu tiên của đường dẫn chính
    X = Conv2D(filters = F1 , kernel_size = (1,1) , strides = (1,1) , padding = 'valid', name = conv_name_base + '2a', kernel_initializer=glorot_uniform(seed = 0))(X)
    #padding = valid thì không thêm viền → kích thước giảm khi kernel lớn hơn 1 và stride bằng 1.
    X = BatchNormalization(axis = 3 , name = bn_name_base+'2a')(X)#Chuẩn hóa activation theo từng channel, giúp quá trình huấn luyện ổn định hơn.
    X = Activation('relu')(X)
    
    # Thành phần thứ hai của lộ trình chính (≈3 dòng)
    X = Conv2D(filters = F2 , kernel_size = (f,f) , strides = (1,1),padding = 'same',name = conv_name_base + '2b', kernel_initializer = glorot_uniform(seed=0))(X)
    #padding = same thì tự động thêm viền → giữ nguyên kích thước không gian khi stride bằng 1.
    X = BatchNormalization(axis = 3, name = bn_name_base + '2b')(X)
    X = Activation('relu')(X)
    
    # Thành phần thứ ba của lộ trình chính (≈2 dòng)
    X = Conv2D(filters = F3 , kernel_size = (1,1), strides = (1,1) , padding = 'valid', name = conv_name_base + '2c', kernel_initializer = glorot_uniform(seed=0))(X)
    X = BatchNormalization(axis = 3 , name = bn_name_base + '2c')(X)
    
    # Bước cuối cùng: Cộng giá trị đường tắt (shortcut) vào đường dẫn chính và truyền qua hàm kích hoạt ReLU (≈2 dòng)
    X = Add()([X,X_shortcut])
    X = Activation('relu')(X)
    
    return X

In [10]:
tf.compat.v1.disable_eager_execution()
tf.compat.v1.reset_default_graph()

with tf.compat.v1.Session() as test:
    np.random.seed(1)
    A_prev = tf.compat.v1.placeholder("float", [3, 4, 4, 6])
    X = np.random.randn(3, 4, 4, 6)
    A = identity_block(A_prev, f = 2, filters = [2, 4, 6], stage = 1, block = 'a')
    test.run(tf.compat.v1.global_variables_initializer())
    out = test.run([A], feed_dict={A_prev: X})
    print("out = " + str(out[0][1][1][0]))

I0000 00:00:1791768246.884224 2959807 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 10101 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060, pci bus id: 0000:01:00.0, compute capability: 8.6
I0000 00:00:1791768247.220373 2995455 cuda_dnn.cc:461] Loaded cuDNN version 92700


out = [0.        0.        1.3454685 2.0318186 0.        1.3246766]


## 2.2 - Khối tích chập (Convolutional block)

"Khối tích chập" (convolutional block) của ResNet là loại khối thứ hai. Bạn có thể sử dụng loại khối này khi kích thước đầu vào và đầu ra không khớp nhau. Điểm khác biệt so với khối đồng nhất (identity block) là có một lớp CONV2D nằm trên đường tắt (shortcut path):

<img src="images/convblock_kiank.png" style="width:650px;height:150px;">
<caption><center> <u> <font color='purple'> **Hình 4** </u><font color='purple'>  : **Khối tích chập** </center></caption>

* Lớp CONV2D trên đường tắt được sử dụng để thay đổi kích thước đầu vào $x$ sang một kích thước khác, sao cho các kích thước khớp nhau tại bước cộng cuối cùng khi cộng giá trị từ đường tắt trở lại đường chính. (Lớp này đóng vai trò tương tự như ma trận $W_s$ đã được thảo luận trong bài giảng.)
* Ví dụ, để giảm chiều cao và chiều rộng của kích thước kích hoạt (activation dimensions) đi một nửa (hệ số 2), bạn có thể sử dụng phép tích chập 1x1 với bước trượt (stride) bằng 2.
* Lớp CONV2D trên đường tắt không sử dụng bất kỳ hàm kích hoạt phi tuyến tính nào. Vai trò chính của nó chỉ là áp dụng một hàm tuyến tính (được học) để giảm kích thước đầu vào, giúp các kích thước khớp nhau cho bước cộng ở giai đoạn sau.

Chi tiết về khối tích chập như sau:

Thành phần đầu tiên của đường chính:
- Lớp CONV2D đầu tiên có $F_1$ bộ lọc (filters) với kích thước (1,1) và bước trượt (s,s). Chế độ padding là "valid" và tên lớp nên là `conv_name_base + '2a'`. Sử dụng giá trị 0 làm hạt giống (seed) cho `glorot_uniform`.
- Lớp BatchNorm đầu tiên thực hiện chuẩn hóa theo trục 'channels' (kênh). Tên lớp nên là `bn_name_base + '2a'`.
- Sau đó áp dụng hàm kích hoạt ReLU. Hàm này không có tên và không có tham số siêu (hyperparameters). Thành phần thứ hai của nhánh chính:
- Lớp CONV2D thứ hai có $F_3$ bộ lọc (filter) với kích thước (f,f) và bước trượt (stride) là (1,1). Chế độ padding là "same" và tên lớp là `conv_name_base + '2b'`. Sử dụng 0 làm giá trị seed cho `glorot_uniform`.
- Lớp BatchNorm thứ hai thực hiện chuẩn hóa theo trục 'channels' (kênh). Tên lớp là `bn_name_base + '2b'`.
- Sau đó áp dụng hàm kích hoạt ReLU. Lớp này không có tên và không có tham số siêu cấp (hyperparameter).

Thành phần thứ ba của nhánh chính:
- Lớp CONV2D thứ ba có $F_3$ bộ lọc với kích thước (1,1) và bước trượt là (1,1). Chế độ padding là "valid" và tên lớp là `conv_name_base + '2c'`. Sử dụng 0 làm giá trị seed cho `glorot_uniform`.
- Lớp BatchNorm thứ ba thực hiện chuẩn hóa theo trục 'channels'. Tên lớp là `bn_name_base + '2c'`. Lưu ý rằng không có hàm kích hoạt ReLU trong thành phần này.

Nhánh tắt (shortcut path):
- Lớp CONV2D có $F_3$ bộ lọc với kích thước (1,1) và bước trượt là (s,s). Chế độ padding là "valid" và tên lớp là `conv_name_base + '1'`. Sử dụng 0 làm giá trị seed cho `glorot_uniform`.
- Lớp BatchNorm thực hiện chuẩn hóa theo trục 'channels'. Tên lớp là `bn_name_base + '1'`.

Bước cuối cùng:
- Cộng giá trị từ nhánh tắt và nhánh chính lại với nhau.
- Sau đó áp dụng hàm kích hoạt ReLU. Lớp này không có tên và không có tham số siêu cấp.

**Bài tập**: Triển khai khối tích chập (convolutional block). Chúng tôi đã triển khai thành phần đầu tiên của nhánh chính; bạn hãy triển khai các phần còn lại. Như trước đây, hãy luôn sử dụng 0 làm giá trị seed cho quá trình khởi tạo ngẫu nhiên để đảm bảo tính nhất quán với hệ thống chấm điểm của chúng tôi. - [Conv2D](https://keras.io/layers/convolutional/#conv2d)
- [BatchNormalization](https://keras.io/layers/normalization/#batchnormalization) (axis: Số nguyên, trục cần được chuẩn hóa (thường là trục đặc trưng))
- Đối với hàm kích hoạt, sử dụng: `Activation('relu')(X)`
- [Add](https://keras.io/layers/merge/#add)

In [ ]:
#convolutional_block: Input và output có thể khác kích thước, nên shortcut cần một lớp convolution để biến đổi input trước khi cộng.
def convolutional_block(X, f, filters, stage, block, s = 2):
    """
    Triển khai khối tích chập (convolutional block) như được mô tả trong Hình 4

Các tham số:
X -- tensor đầu vào có kích thước (m, n_H_prev, n_W_prev, n_C_prev)
f -- số nguyên, xác định kích thước cửa sổ của lớp tích chập (CONV) ở giữa trên nhánh chính
filters -- danh sách các số nguyên (kiểu Python), xác định số lượng bộ lọc (filters) trong các lớp CONV của nhánh chính
stage -- số nguyên, dùng để đặt tên cho các lớp dựa trên vị trí của chúng trong mạng
block -- chuỗi/ký tự, dùng để đặt tên cho các lớp dựa trên vị trí của chúng trong mạng
s -- số nguyên, xác định bước trượt (stride) được sử dụng

Giá trị trả về:
X -- đầu ra của khối tích chập, tensor có kích thước (n_H, n_W, n_C)
    """
    conv_name_base = 'res' + str(stage) + block + '_branch'
    bn_name_base = 'bn' + str(stage) + block + '_branch'
    
    F1 , F2 , F3 = filters # là số lượng filters (bộ lọc) của từng lớp Convolution, đồng thời quyết định số channel đầu ra của lớp đó.
    X_shortcut = X
    #padding = valid thì không thêm viền → kích thước giảm khi kernel lớn hơn 1 và stride bằng 1.
    #Lớp thứ nhất: Dùng s để quyết định có giảm kích thước feature map hay không.
    X = Conv2D(F1 , (1,1) , strides = (s,s) , padding = 'valid' , name = conv_name_base + '2a', kernel_initializer = glorot_uniform(seed=0))(X)
    #Do stride =(s,s) dẫn đến chiều cao và chiều rộng nhỏ hơn
    X = BatchNormalization(axis = 3 , name = bn_name_base + '2a')(X)
    X = Activation('relu')(X)
    
    X = Conv2D(F2 , kernel_size = (f,f), strides=(1,1) , padding ='same',name = conv_name_base + '2b', kernel_initializer = glorot_uniform(seed=0))(X)
    #padding = same thì tự động thêm viền → giữ nguyên kích thước không gian khi stride bằng 1.
    X = BatchNormalization(axis = 3 , name = bn_name_base + '2b')(X)
    X = Activation('relu')(X)
    
    X = Conv2D(F3 , kernel_size = (1,1) , strides =(1,1), padding ='valid', name = conv_name_base + '2c', kernel_initializer = glorot_uniform(seed=0))(X)
    X = BatchNormalization(axis = 3 , name = bn_name_base + '2c')(X)
    
    #Do X và X_shortcut khác kích thước nên phải đưa về cùng shape
    X_shortcut = Conv2D(F3, (1, 1), strides = (s,s), padding = 'valid', name = conv_name_base + '1', kernel_initializer = glorot_uniform(seed=0))(X_shortcut)
    X_shortcut = BatchNormalization(axis = 3, name = bn_name_base + '1')(X_shortcut)
    
    # Final step: Add shortcut value to main path, and pass it through a RELU activation (≈2 lines)
    X = Add()([X, X_shortcut])
    X = Activation('relu')(X) 
    
    ### END CODE HERE ###
    
    return X
    


## 3 - Xây dựng mô hình ResNet đầu tiên của bạn (50 lớp)

Giờ đây, bạn đã có các khối cần thiết để xây dựng một mạng ResNet rất sâu. Hình dưới đây mô tả chi tiết kiến ​​trúc của mạng thần kinh này. "ID BLOCK" trong sơ đồ là viết tắt của "Identity block" (khối đồng nhất), và "ID BLOCK x3" có nghĩa là bạn cần xếp chồng 3 khối đồng nhất lại với nhau.

<img src="images/resnet_kiank.png" style="width:850px;height:150px;">
<caption><center> <u> <font color='purple'> **Hình 5** </u><font color='purple'>  : **Mô hình ResNet-50** </center></caption>

Các chi tiết của mô hình ResNet-50 này bao gồm:

* Zero-padding chèn thêm phần đệm kích thước (3,3) vào đầu vào
* Giai đoạn 1 (Stage 1):
  * Phép tích chập 2D có 64 bộ lọc (filter) kích thước (7,7) và sử dụng bước trượt (stride) là (2,2). Tên của lớp này là "conv1".
  * BatchNorm được áp dụng trên trục 'channels' (kênh) của đầu vào.
  * MaxPooling sử dụng cửa sổ kích thước (3,3) và bước trượt (2,2).
* Giai đoạn 2 (Stage 2):
  * Khối tích chập (convolutional block) sử dụng bộ ba bộ lọc kích thước [64,64,256], "f" là 3, "s" là 1 và tên của khối là "a".
  * 2 khối đồng nhất (identity block) sử dụng bộ ba bộ lọc kích thước [64,64,256], "f" là 3 và tên các khối lần lượt là "b" và "c".
* Giai đoạn 3 (Stage 3):
  * Khối tích chập sử dụng bộ ba bộ lọc kích thước [128,128,512], "f" là 3, "s" là 2 và tên của khối là "a".
  * 3 khối đồng nhất sử dụng bộ ba bộ lọc kích thước [128,128,512], "f" là 3 và tên các khối lần lượt là "b", "c" và "d".
* Giai đoạn 4 (Stage 4):
  * Khối tích chập sử dụng bộ ba bộ lọc kích thước [256, 256, 1024], "f" là 3, "s" là 2 và tên của khối là "a".
  * 5 khối đồng nhất sử dụng bộ ba bộ lọc kích thước [256, 256, 1024], "f" là 3 và tên các khối lần lượt là "b", "c", "d", "e" và "f".
* Giai đoạn 5 (Stage 5):
  * Khối tích chập sử dụng bộ ba bộ lọc kích thước [512, 512, 2048], "f" là 3, "s" là 2 và tên của khối là "a".
  * 2 khối đồng nhất sử dụng bộ ba bộ lọc kích thước [512, 512, 2048], "f" là 3 và tên các khối lần lượt là "b" và "c".
* Lớp Average Pooling 2D sử dụng cửa sổ kích thước (2,2) và có tên là "avg_pool".
* Lớp 'flatten' không có bất kỳ siêu tham số (hyperparameter) hay tên gọi nào.
* Lớp Kết nối đầy đủ (Dense / Fully Connected) thu giảm số lượng đầu ra về đúng số lượng lớp phân loại (classes) bằng cách sử dụng hàm kích hoạt softmax. Tên của lớp này sẽ là `'fc' + str(classes)`.
```[cite: 1]

**Bài tập**: Triển khai mô hình ResNet 50 lớp như được mô tả trong hình trên. Chúng tôi đã triển khai Giai đoạn 1 và 2; bạn hãy triển khai các phần còn lại. (Cú pháp triển khai các Giai đoạn 3-5 sẽ khá tương đồng với Giai đoạn 2.) Hãy đảm bảo tuân thủ quy tắc đặt tên đã nêu ở phần văn bản phía trên. Bạn sẽ cần sử dụng hàm sau:
- Average pooling [xem tài liệu tham khảo](https://keras.io/layers/pooling/#averagepooling2d)

Dưới đây là một số hàm khác mà chúng tôi đã sử dụng trong đoạn mã bên dưới:
- Conv2D: [Xem tài liệu tham khảo](https://keras.io/layers/convolutional/#conv2d)
- BatchNorm: [Xem tài liệu tham khảo](https://keras.io/layers/normalization/#batchnormalization) (axis: Số nguyên, trục cần được chuẩn hóa (thường là trục đặc trưng/features))
- Zero padding: [Xem tài liệu tham khảo](https://keras.io/layers/convolutional/#zeropadding2d)
- Max pooling: [Xem tài liệu tham khảo](https://keras.io/layers/pooling/#maxpooling2d)
- Lớp kết nối đầy đủ (Fully connected layer): [Xem tài liệu tham khảo](https://keras.io/layers/core/#dense)
- Phép cộng (Addition): [Xem tài liệu tham khảo](https://keras.io/layers/merge/#add)

In [ ]:
def ResNet50(input_shape = (64,64,3) , classes = 6) :
    """
    Triển khai kiến ​​trúc ResNet50 phổ biến với cấu trúc sau:
    CONV2D -> BATCHNORM -> RELU -> MAXPOOL -> CONVBLOCK -> IDBLOCK*2 -> CONVBLOCK -> IDBLOCK*3
    -> CONVBLOCK -> IDBLOCK*5 -> CONVBLOCK -> IDBLOCK*2 -> AVGPOOL -> TOPLAYER

    Các tham số:
    input_shape -- kích thước (shape) của các hình ảnh trong tập dữ liệu
    classes -- số nguyên, số lượng lớp (classes)

    Giá trị trả về:
    model -- một đối tượng Model() trong Keras
    """
    X_input = Input(input_shape)
    
    #Zero-Padding
    X = ZeroPadding2D((3,3))(X_input)
    
    #Stage 1
    X = Conv2D(64,(7,7) , strides =(2,2) , name = 'conv1' , kernel_initializer= glorot_uniform(seed = 0))(X)
    X = BatchNormalization(axis = 3 , name = 'bn_conv1')(X)
    X = Activation('relu')(X)
    X = MaxPooling2D((3,3),strides = (2,2))(X)
    
    #Stage 2
    X = convolutional_block(X , f= 3 , filters = [64,64,256] , stage = 2,block = 'a' , s = 1)   
    X = identity_block(X,3,[64,64,256],stage = 2 , block = 'b')
    X = identity_block(X, 3, [64, 64, 256], stage=2, block='c')
    
    # Stage 3
    X = convolutional_block(X, f=3, filters=[128, 128, 512], stage=3, block='a', s=2)
    X = identity_block(X, 3, [128, 128, 512], stage=3, block='b')
    X = identity_block(X, 3, [128, 128, 512], stage=3, block='c')
    X = identity_block(X, 3, [128, 128, 512], stage=3, block='d')

    # Stage 4
    X = convolutional_block(X, f=3, filters=[256, 256, 1024], stage=4, block='a', s=2)
    X = identity_block(X, 3, [256, 256, 1024], stage=4, block='b')
    X = identity_block(X, 3, [256, 256, 1024], stage=4, block='c')
    X = identity_block(X, 3, [256, 256, 1024], stage=4, block='d')
    X = identity_block(X, 3, [256, 256, 1024], stage=4, block='e')
    X = identity_block(X, 3, [256, 256, 1024], stage=4, block='f')

    # Stage 5
    X = convolutional_block(X, f=3, filters=[512, 512, 2048], stage=5, block='a', s=2)
    X = identity_block(X, 3, [512, 512, 2048], stage=5, block='b')
    X = identity_block(X, 3, [512, 512, 2048], stage=5, block='c')
    